# Train Markov Chain + GRU4Rec (co category) tren Kaggle

Notebook nay huan luyen lai 2 mo hinh goi y cho he thong Clickstream Recommendation, dung **toan bo dataset RetailRocket that**, bao gom ca 3 loai event (`view`, `addtocart`, `transaction`) va thong tin **category that**.

## Chuan bi truoc khi chay (lam 1 lan)

1. Tao notebook moi tren Kaggle, upload file nay (`File > Upload Notebook`) hoac copy noi dung vao.
2. O khung ben phai, bam **Add Input** -> tim dataset **`retailrocket/ecommerce-dataset`** -> Add.
3. (Khuyen nghi) Bat **GPU** o `Settings > Accelerator` de train GRU4Rec nhanh hon - **khong can bat Internet**, vi du lieu da duoc dinh kem qua Add Input.
4. Bam **Run All**.

## Noi dung

1. Doc du lieu goc (events, category_tree, item_properties)
2. Cat session theo khoang nghi thoi gian (30 phut) - thay vi gop ca lich su 1 visitor thanh 1 session
3. Trich category that cua tung san pham tu `item_properties`
4. Gan trong so theo loai event (`view` < `addtocart` < `transaction`)
5. Chia train/test theo thoi gian (khong random) de tranh leak du lieu
6. Train Markov Chain (co trong so)
7. Train GRU4Rec (dung ca item + category + loai event lam dau vao)
8. Xay baseline Popularity (toan cuc va theo category cua san pham vua xem)
9. Danh gia Hit Rate@K / MRR cho 4 phuong phap
10. Luu model ra `/kaggle/working/` de tai ve tich hop vao he thong that

In [ ]:
import glob
import os
import pickle
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dang chay tren:", DEVICE)

## Buoc 1 - Doc du lieu goc tu Kaggle Input

Tu dong tim file trong `/kaggle/input/...` du ban dat ten dataset la gi, mien la da Add Input dataset RetailRocket.

In [ ]:
def find_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Khong tim thay {filename} trong /kaggle/input - "
            "hay chac chan ban da Add Input dataset 'retailrocket/ecommerce-dataset'."
        )
    return matches[0]


EVENTS_PATH = find_file("events.csv")
CATEGORY_TREE_PATH = find_file("category_tree.csv")
ITEM_PROPS_PATHS = [find_file("item_properties_part1.csv"), find_file("item_properties_part2.csv")]

print("events.csv:", EVENTS_PATH)
print("category_tree.csv:", CATEGORY_TREE_PATH)
print("item_properties:", ITEM_PROPS_PATHS)

events_df = pd.read_csv(EVENTS_PATH)
events_df["itemid"] = events_df["itemid"].astype(str)
events_df["visitorid"] = events_df["visitorid"].astype(str)
print("So luong event:", len(events_df))
print(events_df["event"].value_counts())

## Buoc 2 - Cat session theo khoang nghi thoi gian

Mot session moi bat dau khi khoang cach giua 2 event lien tiep cua CUNG 1 visitor > 30 phut. Day la cach lam chuan trong bai toan session-based recommendation (giong GRU4Rec goc), thay vi coi ca lich su 1 nguoi la 1 session.

In [ ]:
SESSION_GAP_MS = 30 * 60 * 1000  # 30 phut

events_df = events_df.sort_values(["visitorid", "timestamp"]).reset_index(drop=True)
prev_ts = events_df.groupby("visitorid")["timestamp"].shift(1)
gap = events_df["timestamp"] - prev_ts
is_new_session = gap.isna() | (gap > SESSION_GAP_MS)
session_seq = is_new_session.groupby(events_df["visitorid"]).cumsum()
events_df["session_id"] = events_df["visitorid"] + "_" + session_seq.astype(str)

print("So luong session sau khi cat:", events_df["session_id"].nunique())

## Buoc 3 - Trich category THAT cua tung san pham

`item_properties` la log thay doi thuoc tinh san pham theo thoi gian (dang `timestamp, itemid, property, value`). Property `categoryid` cho biet san pham thuoc category nao - lay gia tri **moi nhat** (theo timestamp lon nhat) cho moi item.

In [ ]:
category_frames = []
for path in ITEM_PROPS_PATHS:
    chunk = pd.read_csv(path)
    chunk = chunk[chunk["property"] == "categoryid"]
    category_frames.append(chunk[["timestamp", "itemid", "value"]])

category_raw = pd.concat(category_frames, ignore_index=True)
category_raw["itemid"] = category_raw["itemid"].astype(str)
category_raw = category_raw.sort_values("timestamp")
item_category = category_raw.groupby("itemid")["value"].last().to_dict()

print("So san pham co category:", len(item_category))
del category_frames, category_raw

## Buoc 4 - Gan trong so theo loai event va chia train/test theo thoi gian

- `view` < `addtocart` < `transaction`: hanh vi cang "manh" thi trong so cang cao khi xay dung ma tran Markov.
- Chia tap test la 15% session **xuat hien sau cung ve thoi gian** (khong random) de mo phong dung tinh huong thuc te: du doan tuong lai tu qua khu, khong de lo thong tin tuong lai vao luc train.

In [ ]:
EVENT_WEIGHT = {"view": 1, "addtocart": 3, "transaction": 5}

session_groups = events_df.groupby("session_id")
session_start_ts = session_groups["timestamp"].min()
cutoff_ts = session_start_ts.quantile(0.85)

sequences = []  # list of session, moi session la list cac (item_id, event_type)
is_train_session = []

for session_id, group in session_groups:
    group = group.sort_values("timestamp")
    items = list(zip(group["itemid"], group["event"]))
    if len(items) < 2:
        continue
    sequences.append(items)
    is_train_session.append(session_start_ts[session_id] < cutoff_ts)

train_sequences = [s for s, is_tr in zip(sequences, is_train_session) if is_tr]
test_sequences = [s for s, is_tr in zip(sequences, is_train_session) if not is_tr]

print(f"Tong so session (>=2 event): {len(sequences)}")
print(f"Train: {len(train_sequences)} session | Test: {len(test_sequences)} session")

## Buoc 5 - Train Markov Chain (co trong so theo loai event)

In [ ]:
transitions = defaultdict(lambda: defaultdict(float))
for session in train_sequences:
    for (item_a, _), (item_b, type_b) in zip(session[:-1], session[1:]):
        transitions[item_a][item_b] += EVENT_WEIGHT[type_b]

markov_matrix = {}
for item_a, next_counts in transitions.items():
    total = sum(next_counts.values())
    markov_matrix[item_a] = {item_b: w / total for item_b, w in next_counts.items()}

print("Markov matrix:", len(markov_matrix), "san pham nguon")
del transitions

## Buoc 6 - Train GRU4Rec (dau vao: item + category + loai event)

Khac voi GRU4Rec co ban (chi dung item_id), model nay ghep them embedding cua **category** va **loai event** o moi buoc, giup model hoc duoc ca "san pham gi", "thuoc nhom gi" va "hanh vi manh hay yeu" cung luc.

In [ ]:
MAX_SEQ_LEN = 20

item2idx = {"<pad>": 0}
cat2idx = {"<pad>": 0, "<unknown>": 1}
event2idx = {"<pad>": 0, "view": 1, "addtocart": 2, "transaction": 3}

for session in train_sequences:
    for item_id, _ in session:
        if item_id not in item2idx:
            item2idx[item_id] = len(item2idx)
        cat_id = item_category.get(item_id, "<unknown>")
        if cat_id not in cat2idx:
            cat2idx[cat_id] = len(cat2idx)

idx2item = {idx: item for item, idx in item2idx.items()}

print(f"So san pham (vocab): {len(item2idx)}")
print(f"So category (vocab): {len(cat2idx)}")

In [ ]:
class GRU4RecCat(nn.Module):
    def __init__(self, n_items, n_categories, n_event_types,
                 item_dim=64, cat_dim=16, event_dim=8, hidden_dim=128):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, item_dim, padding_idx=0)
        self.cat_emb = nn.Embedding(n_categories, cat_dim, padding_idx=0)
        self.event_emb = nn.Embedding(n_event_types, event_dim, padding_idx=0)
        self.gru = nn.GRU(item_dim + cat_dim + event_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, n_items)

    def forward(self, item_seq, cat_seq, event_seq):
        x = torch.cat([
            self.item_emb(item_seq),
            self.cat_emb(cat_seq),
            self.event_emb(event_seq),
        ], dim=-1)
        _, h = self.gru(x)
        return self.output(h.squeeze(0))

In [ ]:
def encode_session(session):
    """Chuyen 1 session (list item_id, event_type) thanh 3 chuoi idx song song."""
    item_idx, cat_idx, event_idx = [], [], []
    for item_id, event_type in session[-MAX_SEQ_LEN - 1:]:
        if item_id not in item2idx:
            continue  # item chi xuat hien o test, bo qua khi encode cho train
        item_idx.append(item2idx[item_id])
        cat_id = item_category.get(item_id, "<unknown>")
        cat_idx.append(cat2idx.get(cat_id, cat2idx["<unknown>"]))
        event_idx.append(event2idx[event_type])
    return item_idx, cat_idx, event_idx


class SessionDataset(Dataset):
    def __init__(self, sessions):
        self.samples = []
        for session in sessions:
            item_idx, cat_idx, event_idx = encode_session(session)
            for t in range(1, len(item_idx)):
                self.samples.append((
                    item_idx[:t], cat_idx[:t], event_idx[:t], item_idx[t],
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item_h, cat_h, event_h, target = self.samples[idx]
        return (
            torch.tensor(item_h, dtype=torch.long),
            torch.tensor(cat_h, dtype=torch.long),
            torch.tensor(event_h, dtype=torch.long),
            target,
        )


def collate(batch):
    item_h, cat_h, event_h, targets = zip(*batch)
    item_h = pad_sequence(item_h, batch_first=True, padding_value=0)
    cat_h = pad_sequence(cat_h, batch_first=True, padding_value=0)
    event_h = pad_sequence(event_h, batch_first=True, padding_value=0)
    return item_h, cat_h, event_h, torch.tensor(targets, dtype=torch.long)


train_dataset = SessionDataset(train_sequences)
print("So mau huan luyen:", len(train_dataset))

In [ ]:
BATCH_SIZE = 1024
EPOCHS = 5
LR = 1e-3

# gioi han so thread CPU khi khong co GPU de tranh qua tai luong lam cham training
if DEVICE.type == "cpu":
    torch.set_num_threads(4)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
print("So batch moi epoch:", len(train_loader))

model = GRU4RecCat(len(item2idx), len(cat2idx), len(event2idx)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch_idx, (item_h, cat_h, event_h, target) in enumerate(train_loader):
        item_h, cat_h, event_h, target = (
            item_h.to(DEVICE), cat_h.to(DEVICE), event_h.to(DEVICE), target.to(DEVICE),
        )
        optimizer.zero_grad()
        logits = model(item_h, cat_h, event_h)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f"  epoch {epoch + 1} batch {batch_idx}/{len(train_loader)}")
    print(f"epoch {epoch + 1}/{EPOCHS} - loss {total_loss / len(train_loader):.4f}")

model.eval()
print("Done training GRU4Rec")

## Buoc 7 - Baseline Popularity (toan cuc va theo category)

Baseline "theo category" dung category cua **san pham vua xem gan nhat** (khong dung thong tin tuong lai) - day la cach trien khai duoc trong thuc te, dung lam fallback khi Markov/GRU4Rec chua co du lieu cho mot item (cold-start).

In [ ]:
global_counter = Counter()
category_counter = defaultdict(Counter)

for session in train_sequences:
    for item_id, event_type in session:
        w = EVENT_WEIGHT[event_type]
        global_counter[item_id] += w
        cat_id = item_category.get(item_id, "<unknown>")
        category_counter[cat_id][item_id] += w

global_popularity = [item for item, _ in global_counter.most_common()]
category_popularity = {
    cat_id: [item for item, _ in counter.most_common()]
    for cat_id, counter in category_counter.items()
}

print("Global popularity:", len(global_popularity), "san pham")
print("So category co bang popularity rieng:", len(category_popularity))

## Buoc 8 - Danh gia Hit Rate@K / MRR cho 4 phuong phap

Ky thuat "last-item masking": voi moi session test, an di item cuoi cung, dung lich su con lai de du doan, kiem tra item that co nam trong top-K khong.

In [ ]:
def category_fallback(last_item_id, k):
    cat_id = item_category.get(last_item_id, "<unknown>")
    return category_popularity.get(cat_id, global_popularity)[:k]


def predict_markov(history, k):
    last_item = history[-1][0]
    if last_item in markov_matrix:
        ranked = sorted(markov_matrix[last_item].items(), key=lambda kv: kv[1], reverse=True)
        return [item for item, _ in ranked[:k]]
    return category_fallback(last_item, k)  # fallback khi item chua tung xuat hien lam nguon


def predict_gru(history, k):
    last_item = history[-1][0]
    item_idx, cat_idx, event_idx = encode_session(history)
    if not item_idx:
        return category_fallback(last_item, k)
    with torch.no_grad():
        item_t = torch.tensor([item_idx], dtype=torch.long).to(DEVICE)
        cat_t = torch.tensor([cat_idx], dtype=torch.long).to(DEVICE)
        event_t = torch.tensor([event_idx], dtype=torch.long).to(DEVICE)
        logits = model(item_t, cat_t, event_t)
        logits[0, 0] = float("-inf")  # bo qua <pad>
        topk = torch.topk(logits, k=min(k, logits.shape[-1] - 1), dim=-1).indices[0].tolist()
    preds = [idx2item[i] for i in topk if i in idx2item]
    return preds if preds else category_fallback(last_item, k)


def predict_global_popular(history, k):
    return global_popularity[:k]


def predict_category_popular(history, k):
    last_item = history[-1][0]
    return category_fallback(last_item, k)


def hit_rate_at_k(predict_fn, sessions, k):
    hits = 0
    for session in sessions:
        history, (true_next, _) = session[:-1], session[-1]
        preds = predict_fn(history, k)
        if true_next in preds:
            hits += 1
    return hits / len(sessions)


def mrr(predict_fn, sessions, k=20):
    scores = []
    for session in sessions:
        history, (true_next, _) = session[:-1], session[-1]
        preds = predict_fn(history, k)
        rank = preds.index(true_next) + 1 if true_next in preds else 0
        scores.append(1 / rank if rank > 0 else 0)
    return sum(scores) / len(scores)

In [ ]:
methods = {
    "Markov Chain (+ category fallback)": predict_markov,
    "GRU4Rec (+ category fallback)": predict_gru,
    "Most Popular (toan cuc)": predict_global_popular,
    "Most Popular (theo category vua xem)": predict_category_popular,
}

rows = []
for name, fn in methods.items():
    row = {"method": name}
    for k in (5, 10, 20):
        row[f"HitRate@{k}"] = round(hit_rate_at_k(fn, test_sequences, k), 4)
    row["MRR"] = round(mrr(fn, test_sequences), 4)
    rows.append(row)

comparison_df = pd.DataFrame(rows)
comparison_df

## Buoc 9 - Luu model de tai ve tich hop vao he thong that

Sau khi chay xong, vao panel **Output** cua notebook, tai cac file ben duoi ve va dat vao thu muc `models/` cua project (de thay the cac file cu).

In [ ]:
OUT_DIR = "/kaggle/working"

with open(f"{OUT_DIR}/markov.pkl", "wb") as f:
    pickle.dump(markov_matrix, f)

torch.save(model.state_dict(), f"{OUT_DIR}/gru4rec_cat.pt")

with open(f"{OUT_DIR}/item2idx.pkl", "wb") as f:
    pickle.dump(item2idx, f)
with open(f"{OUT_DIR}/cat2idx.pkl", "wb") as f:
    pickle.dump(cat2idx, f)
with open(f"{OUT_DIR}/event2idx.pkl", "wb") as f:
    pickle.dump(event2idx, f)
with open(f"{OUT_DIR}/item2category.pkl", "wb") as f:
    pickle.dump(item_category, f)
with open(f"{OUT_DIR}/global_popularity.pkl", "wb") as f:
    pickle.dump(global_popularity, f)
with open(f"{OUT_DIR}/category_popularity.pkl", "wb") as f:
    pickle.dump(category_popularity, f)

comparison_df.to_csv(f"{OUT_DIR}/eval_comparison.csv", index=False)

print("Da luu toan bo file vao", OUT_DIR)
print(os.listdir(OUT_DIR))

## Ket luan va buoc tiep theo

- Model moi dung ca **3 loai event co trong so** va **category that**, train/test chia theo thoi gian, co fallback theo category cho truong hop cold-start.
- File can tai ve: `markov.pkl`, `gru4rec_cat.pt`, `item2idx.pkl`, `cat2idx.pkl`, `event2idx.pkl`, `item2category.pkl`, `global_popularity.pkl`, `category_popularity.pkl`.
- **Luu y quan trong:** kien truc GRU4Rec moi nhan 3 dau vao (item, category, event) thay vi 1 (chi item) nhu ban cu trong `models/gru4rec.py`. File `streaming/inference.py` cua he thong that can duoc cap nhat tuong ung truoc khi dung model nay (doi `GRU4Rec` thanh `GRU4RecCat`, truyen them category_seq/event_seq khi inference). Buoc nay chua lam trong notebook nay - can thuc hien o repo chinh sau khi co file model.